In [15]:
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

# 1. Résolution du problème d'import (le double retour en arrière)
# Comme le notebook est dans GLC-2021/GLC/notebooks/
root_path = os.path.abspath("../..")
if root_path not in sys.path:
    sys.path.append(root_path)

# 2. Import de TA classe de dataset
from GLC.data_loading.pytorch_dataset import GeoLifeCLEF2021Dataset

# 3. Chemins vers les données (ceux qui ont fonctionné)
DATA_PATH = Path(r"C:\Users\abdou\Downloads\geolifeclef-2021\data")
PATCHES_PATH = DATA_PATH / "patches_sample"

# 4. Configuration matérielle
# On vérifie si une carte graphique est dispo, sinon on bascule sur le processeur
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Appareil utilisé pour l'entraînement : {device}")

# 5. Initialisation des datasets (sans les rasters pour le premier test)
print("\nInitialisation des datasets...")
train_dataset = GeoLifeCLEF2021Dataset(root=DATA_PATH, subset="train", use_rasters=False)
val_dataset = GeoLifeCLEF2021Dataset(root=DATA_PATH, subset="val", use_rasters=False)

# 6. Création des DataLoaders
# Un batch_size de 16 est un bon compromis pour commencer sans faire planter la machine
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f" Prêt ! Nombre de lots (batches) pour l'entraînement : {len(train_loader)}")

Appareil utilisé pour l'entraînement : cpu

Initialisation des datasets...
 Prêt ! Nombre de lots (batches) pour l'entraînement : 114580


L'ID d'espèce le plus élevé est 31186.
On redéfinit le modèle avec 31187 classes de sortie.


In [16]:
# 1. Trouver les IDs qui existent VRAIMENT dans ton dossier 'patches_sample'
valid_files = list(PATCHES_PATH.rglob("*_rgb.jpg"))
valid_ids = set([int(f.stem.split('_')[0]) for f in valid_files])

print(f"Nombre d'images réellement présentes sur ton disque : {len(valid_ids)}")

# 2. Filtrer le dataset d'entraînement pour ne garder que ce que tu as
train_indices = [i for i, obs_id in enumerate(train_dataset.observation_ids) if obs_id in valid_ids]
train_dataset.observation_ids = train_dataset.observation_ids[train_indices]
train_dataset.coordinates = train_dataset.coordinates[train_indices]
if train_dataset.targets is not None:
    train_dataset.targets = train_dataset.targets[train_indices]

# 3. Filtrer le dataset de validation
val_indices = [i for i, obs_id in enumerate(val_dataset.observation_ids) if obs_id in valid_ids]
val_dataset.observation_ids = val_dataset.observation_ids[val_indices]
val_dataset.coordinates = val_dataset.coordinates[val_indices]
if val_dataset.targets is not None:
    val_dataset.targets = val_dataset.targets[val_indices]

# 4. Recréer les DataLoaders avec les données filtrées
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

print(f" Nouveau nombre de lots (batches) pour l'entraînement : {len(train_loader)}")

Nombre d'images réellement présentes sur ton disque : 14648
 Nouveau nombre de lots (batches) pour l'entraînement : 873


In [18]:
import torch.nn as nn
import torch.nn.functional as F

# 1. Définition de l'architecture du CNN
class SimpleGeoLifeCNN(nn.Module):
    def __init__(self, num_classes):
        super(SimpleGeoLifeCNN, self).__init__()
        # Le dataset fournit 6 canaux : RGB (3) + NIR (1) + Altitude (1) + Landcover (1)
        self.conv1 = nn.Conv2d(in_channels=6, out_channels=32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Calcul de la taille après les convolutions et le pooling :
        # L'image de base fait 256x256. 
        # Après le 1er MaxPool : 128x128. Après le 2ème : 64x64.
        self.fc1 = nn.Linear(64 * 64 * 64, 512)
        self.fc2 = nn.Linear(512, num_classes)

    def forward(self, x):
        # Passage dans les couches de convolution
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        
        # Aplatissement (Flatten) pour passer aux couches linéaires
        x = x.view(x.size(0), -1) 
        
        # Passage dans les couches denses
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# 2. Instanciation du modèle
# Pour l'instant, mettons un nombre de classes générique (ex: 1000 espèces)
model = SimpleGeoLifeCNN(num_classes=1000).to(device)
print(model)

# 3. Test de validation (Forward Pass) sur le tout premier lot
print("\nTest de passage des données dans le modèle...")
data_iterator = iter(train_loader)
patches, targets = next(data_iterator)

# On envoie les données sur le CPU (ou GPU si disponible plus tard)
patches = patches.to(device)

# On passe les images dans le modèle
outputs = model(patches)

print(f"Forme des données en entrée (batch_size, canaux, hauteur, largeur) : {patches.shape}")
print(f"Forme des prédictions en sortie (batch_size, num_classes) : {outputs.shape}")
print(" Le modèle accepte les données correctement !")

SimpleGeoLifeCNN(
  (conv1): Conv2d(6, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=262144, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=1000, bias=True)
)

Test de passage des données dans le modèle...
Forme des données en entrée (batch_size, canaux, hauteur, largeur) : torch.Size([16, 6, 256, 256])
Forme des prédictions en sortie (batch_size, num_classes) : torch.Size([16, 1000])
 Le modèle accepte les données correctement !


In [ ]:
import torch.optim as optim
import time


# 1. Trouver le plus grand ID d'espèce dans tes données filtrées
max_species_id = int(train_dataset.targets.max())
# On ajoute +1 car si le plus grand ID est 1007, il faut 1008 classes (de 0 à 1007)
vrai_num_classes = max_species_id + 1 

print(f"L'ID d'espèce le plus élevé est {max_species_id}.")
print(f"On redéfinit le modèle avec {vrai_num_classes} classes de sortie.")

# 2. On recrée le modèle avec la bonne taille
model = SimpleGeoLifeCNN(num_classes=vrai_num_classes).to(device)

# 3. TRÈS IMPORTANT : On recrée l'optimiseur pour qu'il se connecte au NOUVEAU modèle
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 1. Configuration de l'apprentissage
# Fonction de perte pour de la classification multiclasse
criterion = nn.CrossEntropyLoss() 
# Optimiseur Adam (ajuste les poids du modèle)
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Nombre d'époques (passages complets sur tes données filtrées)
num_epochs = 3

print(" Début de l'entraînement...")

# 2. Boucle principale
for epoch in range(num_epochs):
    model.train() # Passe le modèle en mode entraînement
    running_loss = 0.0
    start_time = time.time()
    
    for i, (inputs, labels) in enumerate(train_loader):
        # Envoi des données sur le CPU
        inputs, labels = inputs.to(device), labels.to(device)
        
        # Étape A : Remise à zéro des gradients
        optimizer.zero_grad()
        
        # Étape B : Forward pass (prédictions)
        outputs = model(inputs)
        
        # Étape C : Calcul de l'erreur (Loss)
        loss = criterion(outputs, labels)
        
        # Étape D : Backward pass (calcul des gradients)
        loss.backward()
        
        # Étape E : Optimisation (mise à jour des poids)
        optimizer.step()
        
        # Statistiques
        running_loss += loss.item()
        
        # Affichage tous les 5 lots pour voir que ça ne plante pas
        if i % 5 == 0:
            print(f"Époque [{epoch+1}/{num_epochs}] - Lot [{i}/{len(train_loader)}] - Perte (Loss): {loss.item():.4f}")
            
    # Fin d'une époque
    epoch_duration = time.time() - start_time
    epoch_loss = running_loss / len(train_loader)
    print(f" Époque {epoch+1} terminée en {epoch_duration:.2f}s | Perte moyenne : {epoch_loss:.4f}\n")

print(" Entraînement terminé !")

L'ID d'espèce le plus élevé est 31186.
On redéfinit le modèle avec 31187 classes de sortie.
🚀 Début de l'entraînement...
Époque [1/2] - Lot [0/873] - Perte (Loss): 47.5693
Époque [1/2] - Lot [5/873] - Perte (Loss): 2090.1697
Époque [1/2] - Lot [10/873] - Perte (Loss): 30.8468
Époque [1/2] - Lot [15/873] - Perte (Loss): 10.3040
Époque [1/2] - Lot [20/873] - Perte (Loss): 10.3475
Époque [1/2] - Lot [25/873] - Perte (Loss): 10.3429
Époque [1/2] - Lot [30/873] - Perte (Loss): 10.3364
Époque [1/2] - Lot [35/873] - Perte (Loss): 10.3519
Époque [1/2] - Lot [40/873] - Perte (Loss): 10.3295
Époque [1/2] - Lot [45/873] - Perte (Loss): 10.3503
Époque [1/2] - Lot [50/873] - Perte (Loss): 10.3407
Époque [1/2] - Lot [55/873] - Perte (Loss): 10.3480
Époque [1/2] - Lot [60/873] - Perte (Loss): 10.3338
Époque [1/2] - Lot [65/873] - Perte (Loss): 10.3440
Époque [1/2] - Lot [70/873] - Perte (Loss): 10.3313
Époque [1/2] - Lot [75/873] - Perte (Loss): 10.3166
Époque [1/2] - Lot [80/873] - Perte (Loss): 10.

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class BimodalGeoLifeCNN(nn.Module):
    def __init__(self, num_classes):
        super(BimodalGeoLifeCNN, self).__init__()
        
        # Le pooling sera commun aux deux branches pour diviser la taille par 2 à chaque fois
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # =====================================================================
        # BRANCHE 1 : Visuelle (RGB + Near-IR -> 4 canaux)
        # =====================================================================
        self.conv1_vis = nn.Conv2d(in_channels=4, out_channels=16, kernel_size=3, padding=1)
        self.conv2_vis = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.conv3_vis = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        # Taille après 3 pools (256 -> 128 -> 64 -> 32) : 64 filtres de 32x32 pixels
        self.taille_visuelle_apres_conv = 64 * 32 * 32  # = 65536

        # =====================================================================
        # BRANCHE 2 : Topographique (Altitude + Landcover -> 2 canaux)
        # =====================================================================
        self.conv1_topo = nn.Conv2d(in_channels=2, out_channels=16, kernel_size=3, padding=1)
        self.conv2_topo = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.conv3_topo = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, padding=1)
        # Taille après 3 pools : 32 filtres de 32x32 pixels
        self.taille_topo_apres_conv = 32 * 32 * 32      # = 32768

        # =====================================================================
        # FUSION (Late Fusion)
        # =====================================================================
        taille_fusion = self.taille_visuelle_apres_conv + self.taille_topo_apres_conv # = 98304
        
        # Couches denses (Fully Connected) pour la décision finale
        self.fc1 = nn.Linear(taille_fusion, 512)
        self.dropout = nn.Dropout(0.3) # On ajoute un peu de dropout pour éviter l'overfitting
        self.fc2 = nn.Linear(512, num_classes)

    def forward(self, x):
        # 1. SÉPARATION DES MODALITÉS
        # x a pour forme (batch_size, 6, 256, 256)
        x_vis = x[:, :4, :, :]  # Les 4 premiers canaux (RGB + NIR)
        x_topo = x[:, 4:, :, :] # Les 2 derniers canaux (Altitude + Landcover)

        # 2. PASSAGE DANS LA BRANCHE VISUELLE
        out_vis = self.pool(F.relu(self.conv1_vis(x_vis)))
        out_vis = self.pool(F.relu(self.conv2_vis(out_vis)))
        out_vis = self.pool(F.relu(self.conv3_vis(out_vis)))
        out_vis = out_vis.view(out_vis.size(0), -1) # Aplatissement

        # 3. PASSAGE DANS LA BRANCHE TOPOGRAPHIQUE
        out_topo = self.pool(F.relu(self.conv1_topo(x_topo)))
        out_topo = self.pool(F.relu(self.conv2_topo(out_topo)))
        out_topo = self.pool(F.relu(self.conv3_topo(out_topo)))
        out_topo = out_topo.view(out_topo.size(0), -1) # Aplatissement

        # 4. FUSION DES DEUX REPRÉSENTATIONS
        # On colle les vecteurs côte à côte
        out = torch.cat((out_vis, out_topo), dim=1)

        # 5. CLASSIFICATION FINALE
        out = F.relu(self.fc1(out))
        out = self.dropout(out)
        out = self.fc2(out)

        return out

In [10]:
# On instancie le nouveau modèle bimodal (en gardant vrai_num_classes qu'on a calculé avant)
model = BimodalGeoLifeCNN(num_classes=vrai_num_classes).to(device)

# On n'oublie pas de donner les nouveaux paramètres à l'optimiseur !
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Tu peux ensuite relancer ta boucle d'entraînement (for epoch in range...)
for epoch in range(num_epochs):
    model.train() # Passe le modèle en mode entraînement
    running_loss = 0.0
    start_time = time.time()
    
    for i, (inputs, labels) in enumerate(train_loader):
        # Envoi des données sur le CPU
        inputs, labels = inputs.to(device), labels.to(device)
        
        # Étape A : Remise à zéro des gradients
        optimizer.zero_grad()
        
        # Étape B : Forward pass (prédictions)
        outputs = model(inputs)
        
        # Étape C : Calcul de l'erreur (Loss)
        loss = criterion(outputs, labels)
        
        # Étape D : Backward pass (calcul des gradients)
        loss.backward()
        
        # Étape E : Optimisation (mise à jour des poids)
        optimizer.step()
        
        # Statistiques
        running_loss += loss.item()
        
        # Affichage tous les 5 lots pour voir que ça ne plante pas
        if i % 5 == 0:
            print(f"Époque [{epoch+1}/{num_epochs}] - Lot [{i}/{len(train_loader)}] - Perte (Loss): {loss.item():.4f}")
            
    # Fin d'une époque
    epoch_duration = time.time() - start_time
    epoch_loss = running_loss / len(train_loader)
    print(f" Époque {epoch+1} terminée en {epoch_duration:.2f}s | Perte moyenne : {epoch_loss:.4f}\n")

print(" Entraînement terminé !")

Époque [1/2] - Lot [0/873] - Perte (Loss): 14.1034
Époque [1/2] - Lot [5/873] - Perte (Loss): 29.3731
Époque [1/2] - Lot [10/873] - Perte (Loss): 14.7676
Époque [1/2] - Lot [15/873] - Perte (Loss): 10.7408
Époque [1/2] - Lot [20/873] - Perte (Loss): 10.3332
Époque [1/2] - Lot [25/873] - Perte (Loss): 10.3277
Époque [1/2] - Lot [30/873] - Perte (Loss): 10.4002
Époque [1/2] - Lot [35/873] - Perte (Loss): 10.3394
Époque [1/2] - Lot [40/873] - Perte (Loss): 10.3415
Époque [1/2] - Lot [45/873] - Perte (Loss): 10.3610
Époque [1/2] - Lot [50/873] - Perte (Loss): 10.4235
Époque [1/2] - Lot [55/873] - Perte (Loss): 10.3799
Époque [1/2] - Lot [60/873] - Perte (Loss): 10.2970
Époque [1/2] - Lot [65/873] - Perte (Loss): 10.2957
Époque [1/2] - Lot [70/873] - Perte (Loss): 10.1584
Époque [1/2] - Lot [75/873] - Perte (Loss): 10.6208
Époque [1/2] - Lot [80/873] - Perte (Loss): 10.0666
Époque [1/2] - Lot [85/873] - Perte (Loss): 10.2185
Époque [1/2] - Lot [90/873] - Perte (Loss): 9.9424
Époque [1/2] - 

In [13]:
import matplotlib.pyplot as plt

# Permet d'afficher les graphiques directement dans ton notebook
%matplotlib inline

# On compte combien d'époques ont été faites (2 dans ton cas)
epochs_range = range(1, len(history_uni['val_loss']) + 1)

# Création d'une figure avec 2 graphiques côte à côte
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# ==========================================
# GRAPHIQUE 1 : La Perte (Loss)
# (L'objectif est d'être le plus BAS possible)
# ==========================================
ax1.plot(epochs_range, history_uni['val_loss'], label='Unimodal', marker='o', color='#1f77b4', linewidth=2)
ax1.plot(epochs_range, history_bi['val_loss'], label='Bimodal', marker='s', color='#ff7f0e', linewidth=2)
ax1.set_title('Comparaison de l\'Erreur (Validation Loss)')
ax1.set_xlabel('Époques')
ax1.set_ylabel('Perte (Loss)')
ax1.set_xticks(epochs_range)
ax1.legend()
ax1.grid(True, linestyle='--', alpha=0.7)

# ==========================================
# GRAPHIQUE 2 : La Précision (Accuracy)
# (L'objectif est d'être le plus HAUT possible)
# ==========================================
ax2.plot(epochs_range, history_uni['val_accuracy'], label='Unimodal', marker='o', color='#1f77b4', linewidth=2)
ax2.plot(epochs_range, history_bi['val_accuracy'], label='Bimodal', marker='s', color='#ff7f0e', linewidth=2)
ax2.set_title('Comparaison de la Précision (Validation Accuracy)')
ax2.set_xlabel('Époques')
ax2.set_ylabel('Précision (%)')
ax2.set_xticks(epochs_range)
ax2.legend()
ax2.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

NameError: name 'history_uni' is not defined

In [12]:
import os
from pathlib import Path
import torch

# 1. On définit explicitement le chemin de base pour ce notebook
BASE_PATH = Path(r"C:\Users\abdou\Downloads\geolifeclef-2021")
MODELS_PATH = BASE_PATH / "models"

# Création du dossier s'il n'existe pas déjà
os.makedirs(MODELS_PATH, exist_ok=True)

# 2. Sauvegarde du modèle Unimodal
chemin_uni = MODELS_PATH / "cnn_unimodal.pth"
torch.save(model_uni.state_dict(), chemin_uni)
print(f"💾 Mémoire du modèle Unimodal sauvegardée avec succès : {chemin_uni}")

# 3. Sauvegarde du modèle Bimodal
chemin_bi = MODELS_PATH / "cnn_bimodal.pth"
torch.save(model_bi.state_dict(), chemin_bi)
print(f"💾 Mémoire du modèle Bimodal sauvegardée avec succès : {chemin_bi}")

NameError: name 'model_uni' is not defined